# Siem Reap Soil Parameters — 03 Model Building

Every target is predicted from **4 inputs** (Easting, Northing, sample top elevation, logged soil type) by 5 models and 2 references, under two tests.

**What you'll learn:**
- how the honest spatial test (Test A) and the easier random split (Test B) are built
- how Optuna tunes each model without seeing the test boreholes
- how long the runs take, and how to run them in parts

**Models we'll test:**
1. **RF** — Random Forest
2. **XGB** — XGBoost
3. **MLP** — neural network
4. **GWRF** — geographically weighted Random Forest
5. **GWXGB** — geographically weighted XGBoost
6. **Depth** (reference, no tuning) — training mean per 1 m depth slice
7. **SoilDepth** (reference, no tuning) — training mean per soil group and 1 m depth slice

**The two tests:**

```
Test A (main, honest): for each outer fold k = 1..5 (V3's 106 m blockCV folds)
    test  = development boreholes in fold k
    train = the other development boreholes
    inner CV = the other 4 outer folds -> Optuna (30 trials; 20 for GW models), inner out-of-fold RMSE
    refit the best set on all training boreholes -> predict fold k

Test B (comparison only): one random 20 % of the boreholes is the test set; tuning on 5 random
    borehole groups inside the other 80 %. Easier, because test boreholes usually have training
    boreholes nearby (as the Phnom Penh and Yangon notebooks).
```

**How to run:** `PILOT = True` first (one target, XGB, fold 1: a few minutes, and it prints the time). Then `PILOT = False`. Every fold is saved; if the PC stops, run again and it continues. `RUN_TARGETS = ["SPT_N"]` runs one target.

**Run time (estimate):** Test A about 20-40 min per target (10 targets: about 4-6 h); Test B about a quarter of that.

## 1. Import Libraries

In [ ]:
import json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import utils as U

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
SOIL_COLORS = {"Clay": "#5B8C3A", "Sand": "#E5B74A", "other": "#9E9E9E", "missing": "#D0D0D0"}

S = U.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
import joblib
from tqdm import tqdm

## 2. Settings for This Run

In [ ]:
PILOT = True           # True: first target, XGB, fold 1 only.  False: everything.
RUN_TARGETS = None     # e.g. ["SPT_N", "Water_Content"] to run some targets first
RUN_TEST_B = True      # also run Test B (random 80:20 split) after Test A

## 3. Load Cleaned Data

In [ ]:
runs_a = U.run_list(S, test="spatial")
runs_b = U.run_list(S, test="random")
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["cv"]["k"] + 1))
if PILOT:
    runs_a, runs_b, outer = [r for r in runs_a if r["model"] == "XGB"][:1], [], [1]
elif RUN_TARGETS:
    runs_a = [r for r in runs_a if r["target"] in RUN_TARGETS]
    runs_b = [r for r in runs_b if r["target"] in RUN_TARGETS]
PRIV_A, PRIV_B = U.out_dir(S, "03_test_A_spatial", private=True), U.out_dir(S, "03_test_B_random", private=True)
for p in (PRIV_A, PRIV_B):
    (p / "checkpoints").mkdir(exist_ok=True)
targets = {r["target"] for r in runs_a + runs_b}
dev = {t: pd.read_csv(U.processed_dir(S) / f"samples_{t}.csv").query("set == 'development'") for t in targets}
pd.DataFrame([{"target": t, "development samples": len(d), "boreholes": d.BH_Code.nunique()} for t, d in dev.items()])

## 4. Test A — Nested Spatial Cross-Validation
One progress bar per run; it moves one step per Optuna trial. Next to it: mean RMSE and R² of the finished outer folds.

In [ ]:
for r in runs_a:
    per_fold = max(U.n_trials(r["model"], S), 1)
    ck = lambda k: PRIV_A / "checkpoints" / f"{r['tag']}_fold{k}.joblib"
    done = {k: U.scores(*joblib.load(ck(k))["predictions"][["y", "y_pred"]].values.T) for k in outer if ck(k).exists()}
    bar = tqdm(total=len(outer) * per_fold, initial=len(done) * per_fold, desc=f"{r['tag']:<30}", mininterval=10,
               bar_format="{desc} {percentage:3.0f}%|{bar:25}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]{postfix}")

    def show():
        if done:
            bar.set_postfix_str(f"folds {len(done)}/{len(outer)} | mean RMSE {np.nanmean([v['rmse'] for v in done.values()]):.2f}"
                                f" R2 {np.nanmean([v['r2'] for v in done.values()]):.3f}")
    show()
    for k in outer:
        if k in done:
            continue
        res = U.run_fold(r, dev[r["target"]], k, S, on_trial=lambda: bar.update(1))
        if r["model"] in U.REFERENCES:
            bar.update(1)
        res.pop("model")
        joblib.dump(res, ck(k), compress=3)
        done[k] = U.scores(res["predictions"].y, res["predictions"].y_pred)
        show()
    bar.close()
print(f"Test A done ({(time.time() - T0) / 3600:.2f} h in this session).")

In [ ]:
if PILOT:
    res = joblib.load(PRIV_A / "checkpoints" / f"{runs_a[0]['tag']}_fold1.joblib")
    print(f"Pilot {runs_a[0]['tag']} fold 1: {res['info']['test_samples']} test samples, {res['info']['seconds'] / 60:.1f} min. "
          f"Full run: {len(U.run_list(S))} runs x 5 folds (RF, MLP and GW models are slower than XGB; references take seconds).")
    print("If this worked: set PILOT = False and run the notebook again.")

## 5. Test B — Random 80:20 Borehole Split (comparison only)

In [ ]:
if RUN_TEST_B and runs_b:
    test_codes, groups = U.random_split(S)
    print(f"Random split: {len(test_codes)} test boreholes, {len(groups)} training boreholes (same for all targets).")
    for r in tqdm(runs_b, desc="Test B runs", mininterval=10):
        ck = PRIV_B / "checkpoints" / f"{r['tag']}.joblib"
        if ck.exists():
            continue
        res = U.run_random(r, dev[r["target"]], S)
        res.pop("model")
        joblib.dump(res, ck, compress=3)
        sc = U.scores(res["predictions"].y, res["predictions"].y_pred)
        tqdm.write(f"{r['tag']:<34} RMSE {sc['rmse']:.2f}  R2 {sc['r2']:.3f}  ({res['info']['seconds'] / 60:.1f} min)")

## 6. Collect the Results

In [ ]:
for name, priv, out_name in (("Test A", PRIV_A, "outer_predictions.csv"), ("Test B", PRIV_B, "random_predictions.csv")):
    files = sorted((priv / "checkpoints").glob("*.joblib"))
    if PILOT or not files:
        continue
    res = [joblib.load(f) for f in files]
    preds = pd.concat([r["predictions"] for r in res], ignore_index=True)
    preds.to_csv(priv / out_name, index=False)
    info = pd.DataFrame([r["info"] for r in res]).drop(columns=["params"])
    info.to_csv(U.out_dir(S, "03_model_building") / f"run_info_{name.replace(' ', '_')}.csv", index=False)
    print(f"{name}: {len(preds)} test predictions from {len(files)} saved runs/folds, "
          f"{info.seconds.sum() / 3600:.2f} h of computing.")
print(f"Finished. Next: notebook 04 (evaluation).")